# Dimensional — Customer Dimension (SCD Type 2)

Build `globalmart.gold.dim_customer` with versioned rows, effective dates, and current flag. Simulates location changes for **6 customers** (close v1, insert v2).

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.customer_dim as customer_dim_module
importlib.reload(customer_dim_module)

from src.dimensional.customer_dim import (
    CustomerDimConfig,
    build_initial_customer_dimension,
    apply_scd2_customer_changes,
    get_customer_version_history,
    run_customer_dimension,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = CustomerDimConfig()
CHANGE_COUNT = 6
print("Loaded from:", customer_dim_module.__file__)

In [ ]:
initial = build_initial_customer_dimension(spark, config)
initial.write.format("delta").mode("overwrite").saveAsTable(config.target_table)
print("Initial rows:", spark.table(config.target_table).count())
display(spark.table(config.target_table).filter("is_current").limit(5))

In [ ]:
scd2_meta = apply_scd2_customer_changes(spark, config, change_count=CHANGE_COUNT)
print(scd2_meta)
sample_id = scd2_meta["change_ids"][0]
display(get_customer_version_history(spark, sample_id, config))

In [ ]:
import json

written = spark.table(config.target_table)
sample_id = scd2_meta["change_ids"][0]

report = {
    "task": "dim_customer_scd2",
    "target_table": config.target_table,
    "row_count_after_initial": written.count() - scd2_meta["versions_added"],
    "row_count_after_scd2": written.count(),
    "current_customers": written.filter("is_current").count(),
    "scd2_simulation": scd2_meta,
    "sample_customer_version_history": {
        "customer_id": sample_id,
        "versions": [row.asDict() for row in get_customer_version_history(spark, sample_id, config).collect()],
    },
    "sk_strategy": "monotonic_sequence_per_version",
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_customer.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)